In [ ]:
from pathlib import Path
import jsonlines
import pandas as pd
import matplotlib.pyplot as plt

models = ['mmbert']
experiments = [
    'baseline_mmbert', 
    'bce_with_mecla', 
    'bce_with_pairwise_mecla',
    'bce_with_grouped_softmax', 
    'bce_with_grouped_softmax_as_penalty',
    ]
eval_metrics = ['eval_micro_f1', 'eval_macro_f1']
labels = [
    'BRCA_NEGATIVO',
    'BRCA_POSITIVO',
    'CIRURGIA',
    'HER2_NEGATIVO',
    'HER2_POSITIVO',
    'POS_MENOPAUSA',
    'PRE_MENOPAUSA',
    'RE_NEGATIVO',
    'RE_POSITIVO',
    'RP_NEGATIVO',
    'RP_POSITIVO',
    'TIPO_HISTOPATOLOGICO',
]

rename_experiments = {
    'baseline_mmbert': 'Baseline',
    'bce_with_mecla': 'BCE + MECLA',
    'bce_with_pairwise_mecla': 'BCE + P-MECLA',
    'bce_with_grouped_softmax': 'BCE + GS',
    'bce_with_grouped_softmax_as_penalty': 'BCE + GSp',
}

rename_metrics = {
    'eval_micro_f1': 'Micro F1',
    'eval_macro_f1': 'Macro F1',
    'BRCA_NEGATIVO_f1_score':'Negative BRCA - F1-score',
    'BRCA_POSITIVO_f1_score':'Positive BRCA - F1-score',
    'CIRURGIA_f1_score':'Surgery - F1-score',
    'HER2_NEGATIVO_f1_score':'Negative HER2 - F1-score',
    'HER2_POSITIVO_f1_score':'Positive HER2 - F1-score',
    'POS_MENOPAUSA_f1_score':'Post-Menopause - F1-score',
    'PRE_MENOPAUSA_f1_score':'Pre-Menopause - F1-score',
    'RE_NEGATIVO_f1_score':'Negative ER - F1-score',
    'RE_POSITIVO_f1_score':'Positive ER - F1-score',
    'RP_NEGATIVO_f1_score':'Negative PR - F1-score',
    'RP_POSITIVO_f1_score':'Positive PR - F1-score',
    'TIPO_HISTOPATOLOGICO_f1_score':'Histopathological type - F1-score',
}

rename_labels = {
    'BRCA_NEGATIVO':'Negative BRCA',
    'BRCA_POSITIVO':'Positive BRCA',
    'CIRURGIA':'Surgery',
    'HER2_NEGATIVO':'Negative HER2',
    'HER2_POSITIVO':'Positive HER2',
    'POS_MENOPAUSA':'Post-Menopause',
    'PRE_MENOPAUSA':'Pre-Menopause',
    'RE_NEGATIVO':'Negative ER',
    'RE_POSITIVO':'Positive ER',
    'RP_NEGATIVO':'Negative PR',
    'RP_POSITIVO':'Positive PR',
    'TIPO_HISTOPATOLOGICO':'Histopathological type',
}

include_label_specific_f1_scores = True

# ====== plotting config ======
OUTDIR = Path("plots/boxplots")
OUTDIR.mkdir(parents=True, exist_ok=True)

PLOT_GLOBAL_METRICS = True
PLOT_ENTITY_METRICS = False

# If True, one figure per metric (clean + paper-friendly).
# If False, group many metrics into pages (helpful when there are many entities).
ONE_FIGURE_PER_METRIC = False
METRICS_PER_FIG = 6  # only used when ONE_FIGURE_PER_METRIC=False

# ====== storage for stats tests etc ======
group_metrics_for_hypothesis_test = {}

# ====== helper ======
def _make_long_df(group_metrics_for_hypothesis_test, model: str, metrics: list[str]) -> pd.DataFrame:
    """
    Convert nested dict:
      group_metrics_for_hypothesis_test[model][exp][metric] -> list[float]
    into long DataFrame with columns: [model, metric, experiment, value].
    """
    rows = []
    for exp, metric_dict in group_metrics_for_hypothesis_test[model].items():
        exp_name = rename_experiments.get(exp, exp)
        for metric in metrics:
            if metric not in metric_dict:
                continue
            for v in metric_dict[metric]:
                rows.append(
                    {
                        "model": model,
                        "metric": metric,
                        "experiment": exp_name,
                        "value": float(v) if v is not None else None,
                    }
                )
    df_long = pd.DataFrame(rows).dropna(subset=["value"])
    return df_long


def _plot_boxplot_single(df_long: pd.DataFrame, model: str, metric: str, outpath: Path) -> None:
    """
    Make a single boxplot figure for one metric.
    Uses plain matplotlib (no seaborn) and no forced colors.
    """
    
    sub = df_long[(df_long["model"] == model) & (df_long["metric"] == metric)].copy()
    if sub.empty:
        print(f"[skip] No data for {model} / {metric}")
        return

    # preserve ordering from experiments list
    ordered_names = [rename_experiments.get(e, e) for e in experiments]
    print(sub['experiment'])
    print(ordered_names)
    sub["experiment"] = pd.Categorical(sub["experiment"], categories=ordered_names, ordered=True)
    sub = sub.sort_values("experiment")


    data = [sub.loc[sub["experiment"] == name, "value"].to_numpy() for name in ordered_names]

    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    ax.boxplot(
        data,
        tick_labels=ordered_names,
        showmeans=False,
        meanline=False,
    )
    ax.set_ylabel(rename_metrics[metric])
    ax.grid(True, axis="y", alpha=0.25)
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    fig.savefig(outpath, dpi=200)
    plt.close(fig)


def _plot_boxplots_paged(df_long: pd.DataFrame, model: str, metrics: list[str], outpath: Path) -> None:
    """
    Create a "page" (single figure) containing multiple boxplots stacked vertically.
    Still matplotlib-only, no seaborn. Good when you have many entity metrics.
    """
    sub = df_long[df_long["model"] == model].copy()
    ordered_names = [rename_experiments.get(e, e) for e in experiments]

    metrics = [m for m in metrics if ((sub["metric"] == m).any())]
    if not metrics:
        print(f"[skip] No metrics to plot for {model}")
        return

    n = len(metrics)
    fig_h = max(3.0, 2.2 * n)
    fig, axes = plt.subplots(nrows=n, ncols=1, figsize=(8.5, fig_h), sharex=True)
    if n == 1:
        axes = [axes]

    for ax, metric in zip(axes, metrics):
        s = sub[sub["metric"] == metric].copy()
        print(s)
        s["experiment"] = pd.Categorical(s["experiment"], categories=ordered_names, ordered=True)
        s = s.sort_values("experiment")
        data = [s.loc[s["experiment"] == name, "value"].to_numpy() for name in ordered_names]

        ax.boxplot(
            data,
            tick_labels=ordered_names,
            showmeans=True,
            meanline=False,
        )
        ax.set_ylabel(metric)
        ax.grid(True, axis="y", alpha=0.25)

    axes[0].set_title(f"{model} — boxplots")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    fig.savefig(outpath, dpi=200)
    plt.close(fig)


# ====== main loop ======
for model in models:
    group_metrics_for_hypothesis_test[model] = {}

    # we’ll also collect which label cols exist in this run
    label_specific_f1_scores_df = None

    for exp in experiments:
        experiments_to_evaluate = list(Path(f'experiments/{exp}').glob('**/metrics_*.jsonl'))

        metrics = []
        for experiment_path in experiments_to_evaluate:
            with jsonlines.open(experiment_path) as reader:
                for obj in reader:
                    metrics.append(obj)

        df = pd.DataFrame(metrics)

        group_metrics_for_hypothesis_test[model][exp] = {}
        for metric in eval_metrics:
            group_metrics_for_hypothesis_test[model][exp][metric] = df[metric].to_list()

        if include_label_specific_f1_scores:
            if exp == 'exp_6_mecla' or 'grouped_softmax' in exp:
                f1_score_per_label_cols = [col for col in df.columns if col.startswith('eval_f1_')]
                label_specific_f1_scores_df = df[f1_score_per_label_cols].copy()
                label_specific_f1_scores_df.columns = [
                    f"{col.split('eval_f1_')[1]}_f1_score" for col in label_specific_f1_scores_df.columns
                ]
                for col in label_specific_f1_scores_df.columns:
                    group_metrics_for_hypothesis_test[model][exp][col] = label_specific_f1_scores_df[col].to_list()
            else:
                label_specific_f1_scores = []
                for row in df.to_dict('records'):
                    label_specific_f1_score = {f"{k}_f1_score": v['f1_score'] for k, v in row['selected_thresholds'].items()}
                    label_specific_f1_scores.append(label_specific_f1_score)

                label_specific_f1_scores_df = pd.DataFrame(label_specific_f1_scores)
                for col in label_specific_f1_scores_df.columns:
                    group_metrics_for_hypothesis_test[model][exp][col] = label_specific_f1_scores_df[col].to_list()

        print(f'Experiment: {rename_experiments[exp]} - {model}')
        print('---')
        print('Global metrics:')
        print(f"Macro F1: {df['eval_macro_f1'].mean():.3f} ± {df['eval_macro_f1'].std():.3f}")
        print(f"Micro F1: {df['eval_micro_f1'].mean():.3f} ± {df['eval_micro_f1'].std():.3f}")
        print('---')
        if label_specific_f1_scores_df is not None:
            print('Per class metrics:')
            for col in label_specific_f1_scores_df.columns:
                print(f"{col:<30}: {label_specific_f1_scores_df[col].mean():.3f} ± {label_specific_f1_scores_df[col].std():.3f}")
        print('\n\n\n')

    # ====== plotting after we loaded all exps for this model ======
    # Decide which per-entity metric columns to plot
    entity_metric_names = [f"{lab}_f1_score" for lab in labels]

    metrics_to_plot = []
    if PLOT_GLOBAL_METRICS:
        metrics_to_plot += eval_metrics
    if PLOT_ENTITY_METRICS:
        metrics_to_plot += entity_metric_names

    df_long = _make_long_df(group_metrics_for_hypothesis_test, model=model, metrics=metrics_to_plot)

    if ONE_FIGURE_PER_METRIC:
        for metric in metrics_to_plot:
            outpath = OUTDIR / f"{model}__{metric}.png"
            _plot_boxplot_single(df_long, model=model, metric=metric, outpath=outpath)
        print(f"[done] Saved boxplots to: {OUTDIR.resolve()}")
    else:
        # global metrics in one page
        if PLOT_GLOBAL_METRICS:
            outpath = OUTDIR / f"{model}__global_metrics.png"
            _plot_boxplots_paged(df_long, model=model, metrics=eval_metrics, outpath=outpath)

        # entities as multiple pages
        if PLOT_ENTITY_METRICS:
            for i in range(0, len(entity_metric_names), METRICS_PER_FIG):
                chunk = entity_metric_names[i : i + METRICS_PER_FIG]
                outpath = OUTDIR / f"{model}__entity_metrics_{i//METRICS_PER_FIG+1:02d}.png"
                _plot_boxplots_paged(df_long, model=model, metrics=chunk, outpath=outpath)

        print(f"[done] Saved boxplots to: {OUTDIR.resolve()}")


In [ ]:
eval_metrics

In [ ]:
from scipy import stats
from itertools import combinations
from cliffs_delta import cliffs_delta
from statsmodels.stats.multitest import multipletests
import numpy as np

from itertools import combinations
from dataclasses import dataclass
from typing import Any, Dict, List, Sequence, Tuple, Optional

import matplotlib.pyplot as plt


@dataclass
class PairwiseStats:
    experiments: List[str]
    delta: np.ndarray              # (n, n) Cliff's delta, antisymmetric, diagonal 0
    p_raw: np.ndarray              # (n, n) raw p-values (only i<j filled; else nan)
    p_adj: np.ndarray              # (n, n) adjusted p-values (only i<j filled; else nan)
    reject: np.ndarray             # (n, n) True/False (only i<j filled; else False)
    effect_size_label: np.ndarray  # (n, n) object array (only i<j filled; else "")


def _as_1d_float_array(x: Any) -> np.ndarray:
    arr = np.asarray(x)
    if arr.ndim != 1:
        arr = arr.reshape(-1)
    return arr.astype(float)


def compute_pairwise_cliffs_delta_and_wilcoxon(
    group_metrics_for_hypothesis_test: Dict[str, Dict[str, Dict[str, Sequence[float]]]],
    model: str,
    metric: str,
    experiments: Sequence[str],
    alpha: float = 0.05,
    correction: str = "bonferroni",
    zero_method: str = "wilcox",
    alternative: str = "two-sided",
) -> PairwiseStats:
    """
    Compute pairwise Cliff's delta and paired Wilcoxon tests across experiments.

    Notes on Wilcoxon:
    - Requires paired samples of same length.
    - If differences are all zero, scipy may raise ValueError; we treat it as p=1.0.
    """
    exps = list(experiments)
    n = len(exps)

    delta = np.zeros((n, n), dtype=float)
    p_raw = np.full((n, n), np.nan, dtype=float)
    p_adj = np.full((n, n), np.nan, dtype=float)
    reject = np.zeros((n, n), dtype=bool)
    effect_size_label = np.full((n, n), "", dtype=object)

    # Collect pairwise tests for correction
    pairs: List[Tuple[int, int]] = []
    raw_ps: List[float] = []

    for i, j in combinations(range(n), 2):
        exp_i, exp_j = exps[i], exps[j]

        a = _as_1d_float_array(group_metrics_for_hypothesis_test[model][exp_i][metric])
        b = _as_1d_float_array(group_metrics_for_hypothesis_test[model][exp_j][metric])

        if a.shape[0] != b.shape[0]:
            raise ValueError(
                f"Paired test requires same length. "
                f"{exp_i} has {a.shape[0]} values, {exp_j} has {b.shape[0]} values."
            )

        # Wilcoxon paired test (a vs b)
        try:
            _, p = stats.wilcoxon(a, b, zero_method=zero_method, alternative=alternative)
            p = float(p)
            if np.isnan(p):
                p = 1.0
        except ValueError:
            # e.g., all differences are zero
            p = 1.0

        # Cliff's delta (a vs b)
        d, size = cliffs_delta(a, b)

        # Fill matrices
        p_raw[i, j] = p
        delta[i, j] = d
        delta[j, i] = -d  # antisymmetric mirror
        effect_size_label[i, j] = str(size)

        pairs.append((i, j))
        raw_ps.append(p)

    # Multiple testing correction on upper-triangle p-values
    if raw_ps:
        rej, padj, _, _ = multipletests(raw_ps, alpha=alpha, method=correction)
        for (i, j), r, pa in zip(pairs, rej, padj):
            p_adj[i, j] = float(pa)
            reject[i, j] = bool(r)

    return PairwiseStats(
        experiments=exps,
        delta=delta,
        p_raw=p_raw,
        p_adj=p_adj,
        reject=reject,
        effect_size_label=effect_size_label,
    )


def plot_cliffs_delta_matrix(
    stats_: PairwiseStats,
    rename_experiments: Optional[Dict[str, str]] = None,
    title: str = "",
    alpha: float = 0.05,
    show_values: bool = True,
    show_stars: bool = True,
    star_text: str = "★",
    figsize: Tuple[float, float] = (9.5, 8.0),
    vmin: float = -1.0,
    vmax: float = 1.0,
    triangle: str = "upper",          # NEW: "upper", "lower", "both"
    masked_color: str = "#EAEAEA",
    show_grid: bool = False,
    grid_linewidth: float = 1.0,
    hide_spines: bool = True,
) -> plt.Figure:
    """
    Plot NxN heatmap of Cliff's delta with significance overlay.

    triangle:
        "upper" → show only upper triangle
        "lower" → show only lower triangle
        "both"  → show full matrix
    """

    exps = stats_.experiments
    n = len(exps)

    if triangle not in {"upper", "lower", "both"}:
        raise ValueError("triangle must be 'upper', 'lower', or 'both'")

    labels = [
        (rename_experiments.get(e, e) if rename_experiments else e)
        for e in exps
    ]

    fig, ax = plt.subplots(figsize=figsize)

    matrix = stats_.delta.copy()

    # --- Mask according to triangle selection ---
    if triangle == "upper":
        mask = np.tril(np.ones_like(matrix, dtype=bool))
        matrix[mask] = np.nan
    elif triangle == "lower":
        mask = np.triu(np.ones_like(matrix, dtype=bool))
        matrix[mask] = np.nan
    # "both" → no masking

    cmap = plt.cm.Blues.copy()
    cmap.set_bad(color=masked_color)

    im = ax.imshow(matrix, vmin=vmin, vmax=vmax, aspect="equal", cmap=cmap)

    # Axis labels
    ax.set_xticks(np.arange(n))
    ax.set_yticks(np.arange(n))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticklabels(labels)

    # Optional grid
    if show_grid:
        ax.set_xticks(np.arange(-0.5, n, 1), minor=True)
        ax.set_yticks(np.arange(-0.5, n, 1), minor=True)
        ax.grid(which="minor", linestyle="-", linewidth=grid_linewidth)
        ax.tick_params(which="minor", bottom=False, left=False)
    else:
        ax.grid(False)
        ax.set_xticks([], minor=True)
        ax.set_yticks([], minor=True)

    if hide_spines:
        for spine in ax.spines.values():
            spine.set_visible(False)

    # Colorbar
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Cliff's delta (row vs column)")

    # --- Annotations ---
    for i in range(n):
        for j in range(n):

            # Skip masked cells
            if triangle == "upper" and i >= j:
                continue
            if triangle == "lower" and i <= j:
                continue

            d = stats_.delta[i, j]

            # significance always defined for i<j in stats_.reject
            if i < j:
                sig = bool(stats_.reject[i, j])
            elif i > j:
                sig = bool(stats_.reject[j, i])
            else:
                sig = False

            if show_values:
                txt = f"{d:+.3f}"
                if show_stars and sig:
                    txt += f"\n{star_text}"
                ax.text(j, i, txt, ha="center", va="center", fontsize=9)

    if title:
        ax.set_title(title)

    fig.tight_layout()
    return fig

alpha_level = 0.05
global_metrics_pvalues = []
for metric in eval_metrics:
    for model in models:
        print(f'Model: {model}')
        print(f'Metric: {metric}')

        # Perform the Friedman test
        statistic, pvalue = stats.friedmanchisquare(
            *[
                group_metrics_for_hypothesis_test[model][exp][metric]
                for exp in experiments
            ]
        )

        print(f"Friedman test statistic: {statistic}")
        print(f"P-value: {pvalue:.5f}")
        print("=="*20)
        current_metric_pvalues = {
            'Metric': rename_metrics[metric],
        }
        current_metric_pvalues['Friedman test statistic'] = statistic
        current_metric_pvalues['P-value'] = f"{ pvalue:.4f}" if pvalue >= 0.0001 else "$<$0.0001"

        for exp in experiments:
            # get mean and std of this metric for this experiment
            mean = np.array(group_metrics_for_hypothesis_test[model][exp][metric]).mean()
            std = np.array(group_metrics_for_hypothesis_test[model][exp][metric]).std()
            current_metric_pvalues[f"{rename_experiments[exp]}"] = f"""{mean:.3f} $\pm$ {std:.3f}"""

        global_metrics_pvalues.append(current_metric_pvalues)

        # Generate combinations of length 2
        combos_iterator = combinations(experiments, 2)

        # Convert the iterator to a list of tuples for display
        combos_list = list(combos_iterator)
        print('Combinations')
        print(combos_list)
        print("=="*20)

        if pvalue >= alpha_level:
            continue
        
        # post hoc with Wilcoxon Signed-Rank Test for each pair and Cliff's Delta
        p_values = []
        test_results = []
        for group1, group2 in combos_list:
            stat, p = stats.wilcoxon(
                group_metrics_for_hypothesis_test[model][group1][metric],
                group_metrics_for_hypothesis_test[model][group2][metric]
                )
            p_values.append(p)
            test_results.append({'Comparison': f"{rename_experiments[group1]} vs {rename_experiments[group2]}", 'Raw P-Value': p})
            d, size = cliffs_delta(
                group_metrics_for_hypothesis_test[model][group1][metric], group_metrics_for_hypothesis_test[model][group2][metric]
                )
            
            print(f"Comparison: {rename_experiments[group1]} vs {rename_experiments[group2]}")
            print(f"Cliff's Delta: {d}")
            print(f"Effect size: {size}")
            print("=="*20)

        reject, corrected_p, _, _ = multipletests(p_values, alpha=0.05, method='bonferroni')
        results_df = pd.DataFrame(test_results)
        results_df['Corrected P-Value'] = corrected_p
        results_df['Reject H0'] = reject
        print(results_df)

        print('\n\n')

        # Compute + plot regardless, but you can skip overlay if Friedman not significant.
        pair_stats = compute_pairwise_cliffs_delta_and_wilcoxon(
            group_metrics_for_hypothesis_test=group_metrics_for_hypothesis_test,
            model=model,
            metric=metric,
            experiments=experiments,
            alpha=0.05,
            correction="bonferroni",
        )

        fig = plot_cliffs_delta_matrix(
            pair_stats,
            rename_experiments=rename_experiments,
            alpha=0.05,
            show_values=True,
            show_stars=True,
            figsize=(10, 9),
            triangle="lower",
            masked_color="#FFFFFF",
            star_text="*",
        )
        plt.savefig(f"plots/confusion_matrices/cliffs_delta_{model}_{metric}.png")
        plt.show()

In [ ]:
from scipy import stats
from itertools import combinations
from cliffs_delta import cliffs_delta
from statsmodels.stats.multitest import multipletests
import numpy as np

from dataclasses import dataclass
from typing import Any, Dict, List, Sequence, Tuple, Optional

import matplotlib.pyplot as plt


@dataclass
class PairwiseStats:
    experiments: List[str]
    delta: np.ndarray              # (n, n) Cliff's delta, antisymmetric, diagonal 0
    p_raw: np.ndarray              # (n, n) raw p-values (only i<j filled; else nan)
    p_adj: np.ndarray              # (n, n) adjusted p-values (only i<j filled; else nan)
    reject: np.ndarray             # (n, n) True/False (only i<j filled; else False)
    effect_size_label: np.ndarray  # (n, n) object array (only i<j filled; else "")


def _as_1d_float_array(x: Any) -> np.ndarray:
    arr = np.asarray(x)
    if arr.ndim != 1:
        arr = arr.reshape(-1)
    return arr.astype(float)


def compute_pairwise_cliffs_delta_and_wilcoxon(
    group_metrics_for_hypothesis_test: Dict[str, Dict[str, Dict[str, Sequence[float]]]],
    model: str,
    metric: str,
    experiments: Sequence[str],
    alpha: float = 0.05,
    correction: str = "bonferroni",
    zero_method: str = "wilcox",
    alternative: str = "two-sided",
) -> PairwiseStats:
    exps = list(experiments)
    n = len(exps)

    delta = np.zeros((n, n), dtype=float)
    p_raw = np.full((n, n), np.nan, dtype=float)
    p_adj = np.full((n, n), np.nan, dtype=float)
    reject = np.zeros((n, n), dtype=bool)
    effect_size_label = np.full((n, n), "", dtype=object)

    pairs: List[Tuple[int, int]] = []
    raw_ps: List[float] = []

    for i, j in combinations(range(n), 2):
        exp_i, exp_j = exps[i], exps[j]

        a = _as_1d_float_array(group_metrics_for_hypothesis_test[model][exp_i][metric])
        b = _as_1d_float_array(group_metrics_for_hypothesis_test[model][exp_j][metric])

        if a.shape[0] != b.shape[0]:
            raise ValueError(
                f"Paired test requires same length. "
                f"{exp_i} has {a.shape[0]} values, {exp_j} has {b.shape[0]} values."
            )

        try:
            _, p = stats.wilcoxon(a, b, zero_method=zero_method, alternative=alternative)
            p = float(p)
            if np.isnan(p):
                p = 1.0
        except ValueError:
            p = 1.0

        d, size = cliffs_delta(a, b)

        p_raw[i, j] = p
        delta[i, j] = d
        delta[j, i] = -d
        effect_size_label[i, j] = str(size)

        pairs.append((i, j))
        raw_ps.append(p)

    if raw_ps:
        rej, padj, _, _ = multipletests(raw_ps, alpha=alpha, method=correction)
        for (i, j), r, pa in zip(pairs, rej, padj):
            p_adj[i, j] = float(pa)
            reject[i, j] = bool(r)

    return PairwiseStats(
        experiments=exps,
        delta=delta,
        p_raw=p_raw,
        p_adj=p_adj,
        reject=reject,
        effect_size_label=effect_size_label,
    )


def plot_cliffs_delta_matrix(
    stats_: PairwiseStats,
    rename_experiments: Optional[Dict[str, str]] = None,
    title: str = "",
    alpha: float = 0.05,
    show_values: bool = True,
    show_stars: bool = True,
    star_text: str = "★",
    figsize: Tuple[float, float] = (9.5, 8.0),
    vmin: float = -1.0,
    vmax: float = 1.0,
    triangle: str = "upper",
    masked_color: str = "#EAEAEA",
    show_grid: bool = False,
    grid_linewidth: float = 1.0,
    hide_spines: bool = True,
    ax: Optional[plt.Axes] = None,          # NEW
    add_colorbar: bool = True,              # NEW
) -> Tuple[plt.Figure, plt.Axes, plt.Axes | None]:
    """
    Plot NxN heatmap of Cliff's delta with significance overlay.

    Returns (fig, ax, im) so the caller can optionally add a shared colorbar.
    """

    exps = stats_.experiments
    n = len(exps)

    if triangle not in {"upper", "lower", "both"}:
        raise ValueError("triangle must be 'upper', 'lower', or 'both'")

    labels = [(rename_experiments.get(e, e) if rename_experiments else e) for e in exps]

    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = ax.figure

    matrix = stats_.delta.copy()

    if triangle == "upper":
        mask = np.tril(np.ones_like(matrix, dtype=bool))
        matrix[mask] = np.nan
    elif triangle == "lower":
        mask = np.triu(np.ones_like(matrix, dtype=bool))
        matrix[mask] = np.nan

    cmap = plt.cm.Blues.copy()
    cmap.set_bad(color=masked_color)

    im = ax.imshow(matrix, vmin=vmin, vmax=vmax, aspect="equal", cmap=cmap)

    ax.set_xticks(np.arange(n))
    ax.set_yticks(np.arange(n))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticklabels(labels)

    if show_grid:
        ax.set_xticks(np.arange(-0.5, n, 1), minor=True, fontsize=10)
        ax.set_yticks(np.arange(-0.5, n, 1), minor=True, fontsize=10)
        ax.grid(which="minor", linestyle="-", linewidth=grid_linewidth)
        ax.tick_params(which="minor", bottom=False, left=False)
    else:
        ax.grid(False)
        ax.set_xticks([], minor=True)
        ax.set_yticks([], minor=True)

    if hide_spines:
        for spine in ax.spines.values():
            spine.set_visible(False)

    cbar = None
    if add_colorbar:
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label("Cliff's delta (row vs column)")

    for i in range(n):
        for j in range(n):
            if triangle == "upper" and i >= j:
                continue
            if triangle == "lower" and i <= j:
                continue

            d = stats_.delta[i, j]

            if i < j:
                sig = bool(stats_.reject[i, j])
            elif i > j:
                sig = bool(stats_.reject[j, i])
            else:
                sig = False

            if show_values:
                txt = f"{d:+.3f}"
                if show_stars and sig:
                    txt += f"\n{star_text}"
                ax.text(j, i, txt, ha="center", va="center", fontsize=10, color='white', fontweight='bold')

    if title:
        ax.set_title(title)

    return fig, ax, im


def plot_cliffs_delta_matrices_grid(
    items: Sequence[Tuple[PairwiseStats, str]],
    rename_experiments: Optional[Dict[str, str]] = None,
    ncols: int = 2,
    figsize_per_panel: Tuple[float, float] = (8.0, 7.0),
    triangle: str = "lower",
    masked_color: str = "#FFFFFF",
    star_text: str = "*",
    vmin: float = -1.0,
    vmax: float = 1.0,
) -> plt.Figure:
    """
    items: list of (PairwiseStats, title)
    """
    k = len(items)
    nrows = int(np.ceil(k / ncols))

    fig_w = figsize_per_panel[0] * ncols
    fig_h = figsize_per_panel[1] * nrows
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(fig_w, fig_h))
    axes = np.atleast_1d(axes).reshape(nrows, ncols)

    last_im = None
    for idx, (pair_stats, title) in enumerate(items):
        r, c = divmod(idx, ncols)
        ax = axes[r, c]
        _, _, im = plot_cliffs_delta_matrix(
            pair_stats,
            rename_experiments=rename_experiments,
            title=title,
            show_values=True,
            show_stars=True,
            triangle=triangle,
            masked_color=masked_color,
            star_text=star_text,
            vmin=vmin,
            vmax=vmax,
            ax=ax,
            add_colorbar=False,  # important: shared colorbar
        )
        last_im = im

    # Turn off any unused axes
    for idx in range(k, nrows * ncols):
        r, c = divmod(idx, ncols)
        axes[r, c].axis("off")

    # Shared colorbar (compact and non-overlapping)
    if last_im is not None:
        cbar = fig.colorbar(
            last_im,
            ax=axes.ravel().tolist(),
            location="right",
            fraction=0.025,   # smaller width
            pad=0.02,         # smaller gap
            aspect=40
        )
        cbar.set_label("Cliff's delta (row vs column)")

    # Better spacing control (instead of only tight_layout)
    fig.subplots_adjust(
        left=0.06,
        right=0.90,   # leave room for colorbar
        top=0.92,
        bottom=0.08,
        wspace=0.2,  # reduce horizontal space between plots
        hspace=0.25,
    )

    return fig


# -------------------- your loop (minimally changed) --------------------

alpha_level = 0.05
global_metrics_pvalues = []

for model in models:
    grid_items = []  # NEW: collect (PairwiseStats, title) for this model

    for metric in eval_metrics:
        print(f"Model: {model}")
        print(f"Metric: {metric}")

        statistic, pvalue = stats.friedmanchisquare(
            *[
                group_metrics_for_hypothesis_test[model][exp][metric]
                for exp in experiments
            ]
        )

        print(f"Friedman test statistic: {statistic}")
        print(f"P-value: {pvalue:.5f}")
        print("==" * 20)

        current_metric_pvalues = {
            "Metric": rename_metrics[metric],
        }
        current_metric_pvalues["Friedman test statistic"] = statistic
        current_metric_pvalues["P-value"] = f"{pvalue:.4f}" if pvalue >= 0.0001 else "$<$0.0001"

        for exp in experiments:
            mean = np.array(group_metrics_for_hypothesis_test[model][exp][metric]).mean()
            std = np.array(group_metrics_for_hypothesis_test[model][exp][metric]).std()
            current_metric_pvalues[f"{rename_experiments[exp]}"] = f"""{mean:.3f} $\pm$ {std:.3f}"""

        global_metrics_pvalues.append(current_metric_pvalues)

        combos_list = list(combinations(experiments, 2))
        print("Combinations")
        print(combos_list)
        print("==" * 20)

        if pvalue < alpha_level:
            # (kept) post hoc prints
            p_values = []
            test_results = []
            for group1, group2 in combos_list:
                stat, p = stats.wilcoxon(
                    group_metrics_for_hypothesis_test[model][group1][metric],
                    group_metrics_for_hypothesis_test[model][group2][metric],
                )
                p_values.append(p)
                
                d, size = cliffs_delta(
                    group_metrics_for_hypothesis_test[model][group1][metric],
                    group_metrics_for_hypothesis_test[model][group2][metric],
                )

                test_results.append(
                    {"Comparison": f"{rename_experiments[group1]} vs {rename_experiments[group2]}", "Raw P-Value": p, "Cliff's Delta": d, "Effect size": size}
                )

                print(f"Comparison: {rename_experiments[group1]} vs {rename_experiments[group2]}")
                print(f"Cliff's Delta: {d}")
                print(f"Effect size: {size}")
                print("==" * 20)

            reject, corrected_p, _, _ = multipletests(p_values, alpha=0.05, method="bonferroni")
            results_df = pd.DataFrame(test_results)
            results_df["Corrected P-Value"] = corrected_p
            results_df["Reject H0"] = reject
            print(results_df)
            print("\n\n")

            results_df[['Comparison', 'Raw P-Value', "Corrected P-Value", "Cliff's Delta", 'Effect size']].to_latex(f'latex/wilcoxon_test_results_{metric}_global.tex', index=False, formatters={'Raw P-Value': "{:.4f}", "Cliff's Delta": "{:.3f}", "Corrected P-Value": "{:.4f}"})

        # Compute pairwise stats (kept)
        pair_stats = compute_pairwise_cliffs_delta_and_wilcoxon(
            group_metrics_for_hypothesis_test=group_metrics_for_hypothesis_test,
            model=model,
            metric=metric,
            experiments=experiments,
            alpha=0.05,
            correction="bonferroni",
        )

        # NEW: collect for grid
        grid_items.append((pair_stats, f"{rename_metrics.get(metric, metric)}"))

    # NEW: plot grid for this model
    fig = plot_cliffs_delta_matrices_grid(
        grid_items,
        rename_experiments=rename_experiments,
        ncols=2,
        figsize_per_panel=(8.0, 7.0),
        triangle="lower",
        masked_color="#FFFFFF",
        star_text="*",
    )
    plt.savefig(f"plots/confusion_matrices/cliffs_delta_grid_{model}.png", dpi=800, 
                bbox_inches='tight', pad_inches=0.1)
    plt.show()

In [ ]:
pd.DataFrame(global_metrics_pvalues, columns=['Metric', 'Baseline', 'BCE + MECLA', 'BCE + P-MECLA', 'BCE + GS', 'BCE + GSp', 'P-value']).to_latex('latex/global_metrics_pvalues.tex', index=False)

In [ ]:
pd.DataFrame(global_metrics_pvalues, columns=['Metric', 'Baseline', 'BCE + MECLA', 'BCE + P-MECLA', 'BCE + GS', 'BCE + GSp', 'Friedman test statistic', 'P-value']).to_latex('latex/friedman_test_results.tex', index=False, formatters={'Friedman test statistic': "{:.3f}"})

In [ ]:
# Per label analysis
from scipy import stats
from itertools import combinations
from cliffs_delta import cliffs_delta
from statsmodels.stats.multitest import multipletests
import numpy as np

per_label_per_exp_f1_scores_and_pvalues = []

for model in models:
    for label in labels:
        metric = f"{label}_f1_score"
        print(f'Model: {model}')
        print(f'Metric: {metric}')

        current_label_values = {
            'Entity': rename_labels[label],
            }

        for exp in experiments:
            print(exp, len(group_metrics_for_hypothesis_test[model][exp][metric]))
            current_label_values[f'{rename_experiments[exp]}'] = f"{np.array(group_metrics_for_hypothesis_test[model][exp][metric]).mean():.3f} $\pm$ {np.array(group_metrics_for_hypothesis_test[model][exp][metric]).std():.3f}"

        # Perform the Friedman test
        statistic, pvalue = stats.friedmanchisquare(
            *[
                group_metrics_for_hypothesis_test[model][exp][metric]
                for exp in experiments
            ]
        )

        print(f"Friedman test statistic: {statistic}")
        print(f"P-value: {pvalue:.5f}")
        print("=="*20)
        current_label_values['Friedman test statistic'] = f"{statistic:.1f}"
        current_label_values['P-value'] = f"{pvalue:.4f}" if pvalue >= 0.0001 else "$<$0.0001"

        per_label_per_exp_f1_scores_and_pvalues.append(current_label_values)

        # Generate combinations of length 2
        combos_iterator = combinations(experiments, 2)

        # Convert the iterator to a list of tuples for display
        combos_list = list(combos_iterator)
        print('Combinations')
        print(combos_list)
        print("=="*20)

        if pvalue >= alpha_level:
            continue
        # post hoc with Wilcoxon Signed-Rank Test for each pair and Cliff's Delta
        p_values = []
        test_results = []
        for group1, group2 in combos_list:
            stat, p = stats.wilcoxon(
                group_metrics_for_hypothesis_test[model][group1][metric],
                group_metrics_for_hypothesis_test[model][group2][metric]
                )
            p_values.append(p)
            test_results.append({'Comparison': f"{rename_experiments[group1]} vs {rename_experiments[group2]}", 'Raw P-Value': p})
            d, size = cliffs_delta(
                group_metrics_for_hypothesis_test[model][group1][metric], group_metrics_for_hypothesis_test[model][group2][metric]
                )
            
            print(f"Comparison: {rename_experiments[group1]} vs {rename_experiments[group2]}")
            print(f"Cliff's Delta: {d}")
            print(f"Effect size: {size}")
            print("=="*20)

        reject, corrected_p, _, _ = multipletests(p_values, alpha=0.05, method='bonferroni')
        results_df = pd.DataFrame(test_results)
        results_df['Corrected P-Value'] = corrected_p
        results_df['Reject H0'] = reject
        print(results_df)

        print('\n\n')

In [ ]:
pd.DataFrame(per_label_per_exp_f1_scores_and_pvalues)

In [ ]:
pd.DataFrame(per_label_per_exp_f1_scores_and_pvalues).drop(['Friedman test statistic'], axis=1).to_latex('latex/per_label_f1_scores.tex', index=False)